In [ ]:
from sctoolbox.utils.jupyter import bgcolor, _compare_version

# change the background of input cells
bgcolor("PowderBlue", select=[2, 5, 7, 9, 11, 14, 16, 18, 20, 26, 28, 30, 32])

nb_name = "Hotspot.ipynb"

_compare_version(nb_name)

# Hotspot
<hr style="border:2px solid black"> </hr>

## 1 - Description

### 1.1 - Hotspot-Analysis

Hotspot identifies genes with informative local expression structure in single-cell data by computing gene-wise autocorrelations on a cell neighborhood graph. Based on these genes, local gene-gene correlations can be calculated and grouped into modules representing putative gene programs.

The aim of this notebook is to detect locally variable genes, calculate local gene-gene correlations, and group correlated genes into modules. These modules can represent putative gene programs that are active in specific cell states, developmental stages, or cell populations.

The main outputs of this notebook are Hotspot autocorrelation results, selected informative genes, gene modules, module scores per cell, summary tables, and visualizations such as UMAP plots and module-level summaries.

**Reference:** DeTomaso, D. & Yosef, N. (2021). *Hotspot identifies informative gene modules across modalities of single-cell genomics*. Cell Systems, 12(5), 446–456. [https://doi.org/10.1016/j.cels.2021.04.005](https://doi.org/10.1016/j.cels.2021.04.005)

**Documentation:** [https://hotspot.readthedocs.io/en/latest/](https://hotspot.readthedocs.io/en/latest/)

### 1.2 - Prerequisites

**Note: This notebook is written for scRNA-seq data and expects gene symbols in `adata.var_names`.**

**Move this notebook into the notebook folder of the respective analysis, for example `rna_analysis/notebooks/`, before using it.**

Requires a preprocessed AnnData object with raw counts stored in a layer, usually `adata.layers["raw"]`, and relevant metadata in `adata.obs` for plotting or subsetting.

This notebook requires a low-dimensional representation of the data to construct the cell neighborhood graph used by Hotspot. PCA and Harmony-corrected PCA can be computed within this notebook if needed. Other latent representations, such as scVI or Scanorama, must already be present in `adata.obsm` if they should be used for Hotspot.

_______

## 2 - Setup

In [ ]:
import sctoolbox
import sctoolbox.utils as utils
import sctoolbox.tools.hotspot as hotspot
import sctoolbox.plotting as pl

import pandas as pd
from sctoolbox import settings
import gseapy as gp

settings.settings_from_config("config.yaml", key="hotspot")

with pd.option_context("display.max.rows", None, "display.max_colwidth", None):
    display(utils.general.get_version_report(report="versions.yml"))

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Set input anndata
clustered_adata = "anndata_4.h5ad"

_________

## 3 - Load anndata

In [ ]:
adata = utils.adata.load_h5ad(clustered_adata)

with pd.option_context("display.max.rows", 5, "display.max.columns", None):
    display(adata)
    display(adata.obs)
    display(adata.var)

## 4 -  Plot UMAP
<hr style="border:2px solid black"> </hr>

This section visualizes the full AnnData object before optional subsetting and Hotspot analysis. The global UMAP provides an overview of the dataset structure and shows how cells are distributed according to the selected condition column.

This helps users inspect sample, condition, batch, or cell population patterns before deciding whether to run Hotspot on the full dataset or on a biologically relevant subset. The UMAP is used for exploratory visualization and does not determine the latent representation used by Hotspot later in the notebook.


In [ ]:
# Columns in adata.obs that can be used to color the UMAPs, with their number of distinct values.
display(
    pd.DataFrame({
        "n_unique": adata.obs.nunique(),
        "dtype": adata.obs.dtypes.astype(str),
    }).rename_axis("adata.obs column")
)

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Global UMAP overview before subsetting

# Column in adata.obs used to color the UMAP plot.
condition_column = "sample"

# Low-dimensional representation used to compute the UMAP if needed.
# Common options are "X_pca", "X_pca_harmony", "X_scVI", or "X_scanorama".
global_umap_use_rep = "X_pca"

# If True, recompute the UMAP even if adata.obsm["X_umap"] already exists.
global_force_recompute_umap = False

# Shared UMAP computation settings.
# These parameters are passed to the UMAP computation step.
umap_kwargs = {
    "min_dist": 0.5,        # Controls how tightly cells are packed together.
    "spread": 1.0,          # Controls the overall spread of the UMAP layout.
    "n_components": 2,      # Number of UMAP dimensions to compute.
    "random_state": 0,      # Random seed for reproducible UMAP results.
}

In [ ]:
hotspot.ensure_umap_for_visualization(
    adata,
    use_rep=global_umap_use_rep,
    force_recompute=global_force_recompute_umap,
    neighbors_key="neighbors_global_umap",
    **umap_kwargs,
)

_ = pl.embedding.plot_embedding(
    adata,
    color=condition_column,
    save="01_hotspot_global_umap.pdf",
    report="01_hotspot_global_umap.png",
)

__________

## 5 - Subset data (optional)
<hr style="border:2px solid black"> </hr>

### Why should I subset?

Hotspot infers modules from the dominant patterns of variation across the cell
neighborhood graph. On a full dataset these patterns often reflect the
differences between cell types, in which case the modules consist largely of
marker genes and resemble the existing clustering. Depending on the dataset,
they can also follow conditions, time points or batches.

Subsetting to a biologically relevant population removes that dominant signal
and lets finer programs surface: variation *within* one cell type, such as cell
cycle, differentiation gradients, stress response or treatment effects.

**Rule of thumb:** subset when you have a specific biological question about a
subpopulation. Run on the full dataset only when you want a broad overview and
accept that the modules will largely follow the dominant pattern, such as cell types.

Examples of a relevant subset: one cell type (`celltype == "B cell"`), one
condition, one timepoint, or a combination of these.

If you subset here or filter genes in section 6, PCA/Harmony, the neighbour
graph and the UMAP are recomputed in sections 7 and 8. The existing embeddings
were computed on all cells and all genes and would be skewed for the reduced
object.

An empty dictionary (`{}`) means no subsetting; the object is left unchanged.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# If empty the Anndata won't be changed
# Example:
# subset_filters = {
#     "sample": [3, 6],
#     "subcelltype": ["A+", "A-"],
# }

subset_filters = {}

# If True, remove unused categories from categorical obs columns after subsetting.
# Set to False to keep the original category structure.
drop_unused_categories = False

In [ ]:
# Subset the celltype an early stage for example
adata_used = utils.adata.obs_subset(
    adata=adata,
    subset_filters=subset_filters,
    drop_unused_categories=drop_unused_categories,
    name="hotspot_subset",
)
adata_used

## 6 - Filter genes by detection frequency
<hr style="border:2px solid black"> </hr>

Minimum number of cells in which a gene must be detected.
Genes that are detected in only a handful of cells carry almost no information
for the neighbourhood statistics Hotspot computes, but they still enter the
multiple testing correction and the gene-gene correlations. Removing them
sharpens the result and shortens the runtime.

How many cells are "a handful" depends entirely on the dataset, so this section
is split into two steps: 6.1 shows the distribution without changing anything,
6.2 applies a threshold chosen from that distribution.

Requires count-like data in a layer, usually `adata.layers["raw"]`.

### 6.1 - Inspect gene detection
<hr style="border:1px solid black"> </hr>

Counts in how many cells each gene is detected (value > 0) and shows the
distribution as a histogram plus a table of candidate thresholds. Nothing is
removed in this step.

If the data was subsetted in section 5, the counts are recomputed for the
subset: values carried over from notebook 2 refer to all cells and would be
systematically too high.

| Parameter | Description | Options | Default |
|:---|:---|:---|:---|
| `gene_detection_layer` | Layer used to count detections. Count-like data recommended. `None` uses `adata.X`. | `None` or String, e.g. `"raw"`, `"counts"` | `"raw"` |
| `thresholds_to_show` | Thresholds listed in the overview table. `None` derives them from the cell number of the current (sub)set (0.1%, 0.25%, 0.5%, 1%, 2%). | `None` or list of integers | `None` |

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Layer used to count in how many cells each gene is detected.
# Count-like data are recommended, e.g. "raw" or "counts".
# Set to None to use adata.X.
gene_detection_layer = "raw"

# Thresholds listed in the overview table.
# None derives them from the cell number of the current (sub)set.
thresholds_to_show = None

In [ ]:
gene_detection, threshold_table = hotspot.inspect_gene_detection(
    adata_used,
    layer_key=gene_detection_layer,
    thresholds_to_show=thresholds_to_show,
    save="01_hotspot_gene_detection_frequency.pdf",
    report="01_hotspot_gene_detection_frequency.png",
)

display(threshold_table)

### 6.2 - Choose a threshold and filter
<hr style="border:1px solid black"> </hr>

**How to read the output of 6.1.** The histogram is strongly left-skewed in
almost every dataset: a large peak of genes detected in very few cells, then a
long tail of genes present across the population. `min_cells` should be placed
at the right edge of that first peak — far enough to drop the noise, not so far
that you lose genes of a small but real subpopulation. The table shows how many
genes each candidate keeps.

**Rule of thumb:** start at 0.1-1% of the cells in the current (sub)set. The log
output of 6.1 prints that range for this dataset. Lower values are the safer
choice when you expect rare cell states, higher values give a cleaner but
coarser gene set. Losing 20-50% of the genes at this step is normal; losing
almost all of them means the threshold is far too high.

Note that this interacts with the gene selection in section 9: filtering here
reduces the pool that `top_n` or the FDR cutoff draw from.

| Parameter | Description | Options | Default |
|:---|:---|:---|:---|
| `min_cells` | Minimum number of cells a gene must be detected in. `None` keeps all genes. Deliberately has no numeric default — choose it from 6.1. | `None` or Integer | `None` |

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Choose this from the histogram and the table in 6.1.
# Set to None to keep all genes.
min_cells = None

In [ ]:
adata_filtered, gene_filter_summary = hotspot.filter_genes_by_cell_detection(
    adata_used,
    min_cells=min_cells,
)

## 7 - Select and calculate Hotspot latent key for hotspot
<hr style="border:2px solid black"> </hr>

### What is a latent representation?

Hotspot asks, for every gene, whether cells that are close to each other also
show similar expression of that gene. "Close" is defined by a KNN graph, and
that graph is not built on the raw genes but on a latent representation, a
compressed description of the data. In a PCA, the expression of thousands of
genes is summarized into e.g. 50 principal components (PCs): each PC is a weighted
combination of many genes, similar to a Hotspot module. The first PC carries
the most variance, the last the least, and every cell gets a value for every
PC. Distances computed on these components are less affected by technical
noise than distances on the raw genes.

### What happens in this step?

The notebook selects one representation from `adata.obsm` and stores its key in
`hotspot_latent_key`. PCA and Harmony-corrected PCA are computed here if they
are missing. Representations from other tools, such as `X_scVI` or
`X_scanorama`, are reused if they already exist but are not computed by this
notebook.
To decide, look at the UMAP from section 4, colored by your batch column
(e.g. the sample). Cells of the same cell type should show distinct populations independent of the batch.
| Representation | When to use it |
|:---|:---|
| `X_pca` | All cells come from one sample, or the batches overlap in the UMAP of section 4. Start here. |
| `X_pca_harmony` | In the UMAP of section 4, cells of the same cell type form separate groups per batch. Needs `batch_key`. |
| `X_scVI`, `X_scanorama` | An integration was already computed in an earlier notebook and you did not subset cells in section 5. Otherwise it still describes the removed cells. |

### Why does this matter?

The representation determines which cells are considered neighbours and therefore directly affects the downstream Hotspot analysis, including the detected autocorrelated genes, gene-gene correlations, and resulting gene modules. A representation that still carries a batch effect produces
modules that follow the batch rather than biology. One that is too coarse
smooths away the programs you are looking for.

Note that this is separate from the UMAP used in Sections 4 and 8. The UMAP is used for visualizing the data, whereas in this context the latent representation refers to the representation on which Hotspot performs its computations.

### Further reading

- Hotspot documentation: https://hotspot.readthedocs.io/en/latest/
- DeTomaso & Yosef, *Hotspot identifies informative gene modules across
  modalities of single-cell genomics*, Cell Systems 12(5), 2021:
  https://doi.org/10.1016/j.cels.2021.04.005
- Harmony (batch correction): Korsunsky et al., *Fast, sensitive and accurate
  integration of single-cell data with Harmony*, Nature Methods 16, 2019:
  https://doi.org/10.1038/s41592-019-0619-0
  

### Parameters

#### Metadata columns

| Parameter | Description | Options / Example | Default |
|:---|:---|:---|:---|
| `condition_column` | Column in `adata.obs` used for plotting and colouring cells. | String, e.g. `"sample"`, `"cell_type"` | `"sample"` |
| `batch_key` | Column in `adata.obs` used for Harmony batch correction. `None` disables batch correction. | `None` or String, e.g. `"sample"`, `"batch"` | `"sample"` |

#### Representation

| Parameter | Description | Options | Default |
|:---|:---|:---|:---|
| `hotspot_rep` | Representation Hotspot builds its neighbor graph on. PCA and Harmony are computed if missing and recomputed after cell subsetting (Section 5) or gene filtering (Section 6); other keys must already exist. See the \"When to use it\" table at the top of this section for which one to choose. | `X_pca`, `X_pca_harmony`, or another key from the list of available representations below, e.g. `X_scVI` | `X_pca_harmony` |
| `PCA_SETTINGS["n_comps"]` | Number of principal components computed when a PCA is (re)computed. Ignored if an existing PCA is reused. | Integer | `50` |
| `PCA_SETTINGS["use_highly_variable"]` | Whether the PCA uses only the genes flagged in `adata.var["highly_variable"]`. `False` uses all genes, which is safer after gene filtering in section 6. | `True`, `False` | `False` |

In [ ]:
# Representations available in adata.obsm, with their number of dimensions.
# 2D embeddings such as X_umap are meant for plotting and not suitable for Hotspot.
display(
    pd.DataFrame(
        {"n_dims": {key: rep.shape[1] for key, rep in adata_filtered.obsm.items()}}
    ).rename_axis("adata.obsm key")
)

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Metadata columns
batch_key = "sample"

# Representation Hotspot builds its neighbor graph on.
#   "X_pca"         -> PCA, computed here if missing
#   "X_pca_harmony" -> Harmony-corrected PCA, computed here if missing; needs batch_key
#   any other key   -> must be one of the keys listed in the table above, e.g. "X_scVI"
hotspot_rep = "X_pca_harmony"

# PCA settings, only used when a PCA is (re)computed
PCA_SETTINGS = {
    "n_comps": 50,               # number of principal components
    "use_highly_variable": False,  # True: PCA only on adata.var["highly_variable"]
}

In [ ]:
hotspot_latent_key = hotspot.ensure_representation(
    adata=adata_filtered,
    use_rep=hotspot_rep,
    batch_key=batch_key,
    force_recompute=bool(subset_filters) or min_cells is not None,
    **PCA_SETTINGS
)

print(f"Selected Hotspot latent representation: {hotspot_latent_key}")

## 8 - (Re-) Calculate UMAP after subsetting
<hr style="border:2px solid black"> </hr>

### Why is the UMAP recomputed?

The UMAP from section 4 was computed on all cells and all genes. If you
subsetted in section 5, that layout no longer describes the data you are
working with: its axes are still influenced by the populations you removed, and
the differences that remain inside your subset get compressed into a small part
of the plot. Filtering genes in section 6 has a smaller but similar effect,
since the PCA the layout is built on combines all genes.

Recomputing gives the subset its own layout, so that structure within the
selected population becomes visible — which is exactly the structure Hotspot
will later report as modules.

If neither cells nor genes were removed, an existing UMAP is reused and nothing
is overwritten.

### What is this UMAP used for?

Visualisation only. The module scores in section 10 are plotted on it, which
makes it easier to see where in the data a module is active.

There are two neighbor graphs in this notebook:

1. the graph for this UMAP, built here with Scanpy on `hotspot_latent_key`,
2. the graph Hotspot computes on, built by Hotspot itself in section 9 on
   `hotspot_latent_key`.

Section 7 only selects or computes the representation; no graph is built
there. PCA/Harmony are not computed again here.

### Parameters

| Parameter | Description | Options | Default |
|:---|:---|:---|:---|
| `force_recompute_umap` | Recompute neighbour graph and UMAP even if neither cells nor genes were removed, e.g. after changing `umap_kwargs`. After subsetting (section 5) or gene filtering (section 6) they are always recomputed. PCA/Harmony are not recomputed here; that happens in section 7. | `True`, `False` | `False` |
| subset_neighbors_key | Key the neighbor graph for this UMAP is stored in adata.uns. A separate key ensures that existing graphs in the object, e.g. the one used for clustering, are not overwritten. | String | "neighbors_hotspot_umap" |

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Recompute neighbour graph and UMAP even if neither cells nor genes were
# removed, e.g. after changing umap_kwargs in section 4. After subsetting
# (section 5) or gene filtering (section 6) they are always recomputed.
force_recompute_umap = False

# Separate key so that this graph does not overwrite the one from section 4.
subset_neighbors_key = "neighbors_hotspot_umap"

In [ ]:
# PCA/Harmony were already (re)computed in section 7; only the neighbour
# graph and the UMAP are rebuilt here, on the representation Hotspot uses.
hotspot.ensure_umap_for_visualization(
    adata_filtered,
    use_rep=hotspot_latent_key,
    batch_key=batch_key,
    force_recompute=bool(subset_filters) or min_cells is not None or force_recompute_umap,
    neighbors_key=subset_neighbors_key,
    use_highly_variable=PCA_SETTINGS["use_highly_variable"],
    **umap_kwargs,
)

_ = pl.embedding.plot_embedding(
    adata_filtered,
    color=condition_column,
    title=f"UMAP after subsetting\ncolored by {condition_column}",
    save="03_hotspot_subset_umap.pdf",
    report="01_hotspot_subset_umap.png",
)

## 9 - Hotspot Run
<hr style="border:2px solid black"> </hr>

### Where to start

If you are unsure, the defaults below are a reasonable first run. The three
parameters worth revisiting afterwards:

- **`n_neighbors`** controls how far Hotspot looks. Too small and modules
  fragment into noise, too large and distinct programs get merged. Start at
  `30`, lower it for small subsets.
- **`gene_selection_method`** decides how many genes enter the correlation
  step, which drives the runtime. `"top_n_z"` with 500 genes is fast and
  usually enough to see the main programs. Switch to `"fdr"` once you know the
  dataset and want completeness.
- **`min_gene_threshold`** sets how granular the modules are. If you get two
  or three very large modules, lower it; if you get many tiny ones, raise it.

None of these has one correct value — run once, look at the module sizes and
the score UMAPs in section 10, and adjust.

### Parameters

Hotspot is not limited to scRNA-seq. For other modalities the layer and the
null model have to match the data — see `model` below and the
[Hotspot documentation on the model choice](https://hotspot.readthedocs.io/en/latest/#alternative-choices-for-model).

#### Input

The layer from section 6.1 (`gene_detection_layer`) and the representation
from section 7 (`hotspot_latent_key`) are reused.

| Parameter | Description | Options | Default |
|:---|:---|:---|:---|
| `umi_counts_obs_key` | Column in `adata.obs` with the total counts per cell. Computed from `layer_key` if missing. | String | `"total_counts"` |
| `model` | Null model for the per-gene distribution. Must match what is in `layer_key`: `"danb"` for UMI counts (scRNA-seq), `"bernoulli"` for binary or very sparse data such as scATAC peak matrices, `"normal"` for continuous normalised data, `"none"` for already variance-stabilised data. [Details](https://hotspot.readthedocs.io/en/latest/#alternative-choices-for-model) | `"danb"`, `"bernoulli"`, `"normal"`, `"none"` | `"danb"` |

#### Neighbour graph

| Parameter | Description | Options | Default |
|:---|:---|:---|:---|
| `n_neighbors` | Number of neighbors in Hotspot's own KNN graph. Hotspot builds this graph itself on `hotspot_latent_key`; it is independent of the Scanpy graph used for the UMAP and is not stored in `adata`. Small values (10–15) pick up fine, local programs but are noisier; large values (50–100) give smoother, broader programs. For subsets below ~1000 cells use 10–15. | Integer | `30` |
| `weighted_graph` | Whether neighbours are weighted by distance. `False` treats all neighbours equally and is more robust. | `True`, `False` | `False` |
| `jobs` | Number of parallel jobs. Taken from `config.yaml`. | Integer | `settings.threads` |

#### Gene selection

| Parameter | Description | Options | Default |
|:---|:---|:---|:---|
| `gene_selection_method` | How genes for the local correlations are chosen. This step compares every pair of selected genes, so its runtime grows with the square of the number of genes. A lower `top_n`, therefore reduces runtime. `"fdr"` takes every significant gene; how many that are depends on the data and can be several thousand, which makes the step much slower. Start with `"top_n_z"`. | `"top_n_z"`, `"fdr"` | `"top_n_z"` |
| `top_n` | Number of genes taken by Z-score. Only used with `"top_n_z"`. | Integer, 500–1000 to start | `500` |
| `autocorr_fdr_threshold` | FDR cutoff for gene selection. Only used with `"fdr"`. | Float | `0.05` |

#### Modules

| Parameter | Description | Options | Default |
|:---|:---|:---|:---|
| `min_gene_threshold` | Minimum number of genes per module. Smaller values give more and smaller modules. | Integer | `20` |
| `core_only` | Hotspot builds modules by hierarchically clustering the locally correlated genes. A gene counts as a **core gene** of a module when it correlates strongly with the module as a whole, not just with a few of its members. With `True` only those genes are assigned and the rest is labelled unassigned (`-1`), which gives smaller but cleaner modules. With `False` genes at the cluster border are assigned as well, giving larger modules with a less specific signature. Keep `True` unless you deliberately want broader gene lists, e.g. for enrichment. | `True`, `False` | `True` |
| `module_fdr_threshold` | FDR cutoff used when modules are created. | Float | `0.05` |

#### Which steps to run

| Parameter | Description | Options | Default |
|:---|:---|:---|:---|
| `compute_local_correlations` | Compute gene-gene correlations for the selected genes. Required for modules. | `True`, `False` | `True` |
| `compute_modules` | Group correlated genes into modules. | `True`, `False` | `True` |
| `compute_module_scores` | Compute a per-cell score for each module. Requires `compute_modules=True`. | `True`, `False` | `True` |

#### Output

| Parameter | Description | Options | Default |
|:---|:---|:---|:---|
| `module_prefix` | Prefix of the module score columns added to `adata.obs`. | String | `"Hotspot module"` |
| `store_local_correlations` | Whether to keep the local correlation matrix in `adata.uns`. It grows with the square of the number of selected genes, so the `.h5ad` gets large. | `True`, `False` | `False` |

<h1><center>⬐ Fill in input data here ⬎</center></h1>

### 9.1 - input hotspot parameters

In [ ]:
# Hotspot input settings
umi_counts_obs_key = "total_counts"

# Null model for the per-gene expression distribution.
# Pick the one that matches the input in layer_key:
#   "danb"      - depth-adjusted negative binomial. For UMI counts, i.e. the
#                 usual choice for scRNA-seq with layer_key="raw".
#   "bernoulli" - models detection (zero vs non-zero) only. For binary or very
#                 sparse data such as scATAC peak matrices, or when counts are
#                 unreliable.
#   "normal"    - for continuous data that is already normalised and
#                 log-transformed, or for non-count modalities.
#   "none"      - no null model; values are only centered and scaled. Use when
#                 the data is already variance-stabilised.
# See https://hotspot.readthedocs.io/en/latest/#alternative-choices-for-model for the underlying statistics.
model = "danb"

# Graph settings
n_neighbors = 15
weighted_graph = False
jobs = settings.threads

# Gene selection
gene_selection_method = "top_n_z"   # available: "fdr", "top_n_z"
autocorr_fdr_threshold = 0.05
top_n = 500

# Module settings
min_gene_threshold = 30
core_only = True
module_fdr_threshold = 0.05

# Outputs
compute_local_correlations = True
compute_modules = True
compute_module_scores = True

module_prefix = "Hotspot module"

# Whether to keep the local correlation matrix in adata.uns.
# It grows with the square of the number of selected genes;
# keep False unless you need it, otherwise the .h5ad gets large.
store_local_correlations = False

### 9.2 - Run hotspot

In [ ]:
results = hotspot.run_hotspot(
    adata=adata_filtered,
    layer_key=gene_detection_layer,
    latent_obsm_key=hotspot_latent_key,
    umi_counts_obs_key=umi_counts_obs_key,
    model=model,
    n_neighbors=n_neighbors,
    weighted_graph=weighted_graph,
    jobs=jobs,
    top_n=top_n,
    gene_selection_method=gene_selection_method,
    autocorr_fdr_threshold=autocorr_fdr_threshold,
    min_gene_threshold=min_gene_threshold,
    core_only=core_only,
    module_fdr_threshold=module_fdr_threshold,
    compute_local_correlations=compute_local_correlations,
    compute_modules=compute_modules,
    compute_module_scores=compute_module_scores,
    module_prefix=module_prefix,
)

_________

## 10 - Locally variable genes and modules
<hr style="border:2px solid black"> </hr>

This section shows what Hotspot found: which genes carry structure across the
cell neighbourhood graph, which of those move together, and how the resulting
gene programs are distributed over the cells.

The four steps build on each other: autocorrelation picks the genes worth
looking at, the local correlations show which of them behave alike, the
modules group them, and the module scores place those groups back on the cells.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# FDR cutoff used to count how many genes show significant autocorrelation.
# Only affects the reported number in 10.1, not the Hotspot run or the gene
# selection, which is controlled by gene_selection_method in section 9.
fdr_threshold = 0.05

### 10.1 - Autocorrelation

For every gene, Hotspot asks whether cells that are neighbours in the latent
representation also show similar expression of that gene. Four statistics come
out of this, and one histogram is shown per statistic.

**What you see.** `C` is the autocorrelation strength, `Z` the corresponding
test statistic, `Pval` its significance and `FDR` the same corrected for the
number of genes tested.

Only the significant genes enter the next step, either by FDR cutoff or as the
top N by Z-score, depending on `gene_selection_method` in section 9.

| Parameter | Description | Options | Default |
|:---|:---|:---|:---|
| `fdr_threshold` | Cutoff used to count how many genes are significant. Only affects the reported number, not the Hotspot run. | Float | `0.05` |

In [ ]:
summary = hotspot.show_hotspot_results(
    adata=adata_filtered,
    fdr_threshold=fdr_threshold,
    plot_autocorrelation_distribution=True,
    plot_module_sizes=False,
    plot_umap_modules=False,
    save_autocorrelation="hotspot_autocorrelation.pdf",
)

with pd.option_context("display.max.rows", 10, "display.max.columns", None):
    display(summary["autocorrelations"])

### 10.2 - Local correlations

The selected genes are correlated against each other, again locally: two genes
count as correlated if they are high in the same neighbourhoods, not if they
are high in the same individual cells. The heatmap shows all pairs, ordered by
the hierarchical clustering Hotspot computed.

In [ ]:
# Needs the Hotspot object and can therefore not be read from adata
hotspot.plot_local_correlations(
    results["hotspot"],
    save="02_hotspot_local_correlations_heatmap.pdf",
    report="02_hotspot_local_correlations_heatmap.png",
)

### 10.3 - Modules

The blocks of the heatmap are cut into modules. Each module is a set of genes
that vary together across the cell neighborhoods, i.e. a candidate gene
program.

Genes without a clear module are designated as `-1` in
`adata.var["hotspot_module"]`, which can be interpreted as n/a. The barplot
shows only the actual modules.

In [ ]:
module_summary = hotspot.show_hotspot_results(
    adata=adata_filtered,
    plot_autocorrelation_distribution=False,
    plot_module_sizes=True,
    plot_umap_modules=False,
    save_module_sizes="03_hotspot_module_sizes.pdf",
    report_module_sizes="03_hotspot_module_sizes.png",
)

display(module_summary["module_sizes"])

### 10.4 - Module scores

Each cell gets one score per module, summarizing how active that gene program
is in the cell. The scores are plotted on the UMAP from section 8.

In [ ]:
_ = hotspot.show_hotspot_results(
    adata=adata_filtered,
    plot_autocorrelation_distribution=False,
    plot_module_sizes=False,
    plot_umap_modules=True,
    save_module_scores="04_hotspot_module_scores_umap.pdf",
    report_module_scores="04_hotspot_module_scores_umap.png",
)

### 10.5 - Module assignment per cell

Hotspot gives every cell a score for every module, not one module per cell.
Many analyses expect groups instead, e.g. marker genes, a comparison with a
clustering, or most other notebooks of this framework. Here each cell is
therefore assigned to the module with the highest score.

The scores are scaled per module first, so that a value of 1 means the same
for every module: one standard deviation above the average of all cells.
Without this, modules whose scores simply have a wider range would win most
cells.

Cells whose activity is not clearly above average in any module can be
labeled unassigned instead of being forced into a module. How strict this
is, is set with module_min_score in 10.5.2, after looking at the
distribution in 10.5.1.

The result is stored in `adata.obs["hotspot_module_assignment"]` and can be
used like a clustering in other notebooks.

### 10.5.1 - How clearly do cells belong to one module?

**What do I see?** For every cell, the highest of its scaled module scores.
The table shows how many cells would be assigned at different thresholds.

**How can I interpret it?** Cells on the right have one clearly active
module. Cells around 0 or below are average in every module; assigning them
to a module would be mostly arbitrary. A peak far to the right means the
modules separate the cells well. If most cells sit around 0, the modules
describe programs that are active in only a few cells. A threshold where the
histogram starts to rise, often between 0.5 and 1, is a good starting point.

In [ ]:
threshold_table = hotspot.inspect_module_assignment(
    adata_filtered,
    save="05_hotspot_module_assignment_distribution.pdf",
    report="05_hotspot_module_assignment_distribution.png",
)
display(threshold_table)

### 10.5.2 - Assign cells to modules

**What do I see?** The UMAP colored by the module each cell was assigned to.

**How can I interpret it?** A module that covers one coherent region of the
UMAP describes a program specific to that population. A module scattered over
the whole UMAP describes a program that is active in many cell types, e.g.
cell cycle or stress. Large `unassigned` regions mean that the activity of
every module there is at or below the average of all cells. This includes
cells with average activity in all modules, not only cells without activity.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Minimum scaled module score a cell needs in its best module, in standard
# deviations above the module mean. Cells below it are labeled "unassigned".
# None assigns every cell, including cells that are not above average in any module.
# Choose the value based on the histogram and table in 10.5.1.
module_min_score = 0.5

In [ ]:
hotspot.assign_cells_to_modules(adata_filtered, min_score=module_min_score)

pl.embedding.plot_embedding(
    adata_filtered,
    method="umap",
    color="hotspot_module_assignment",
    rasterize=True,
    title="Hotspot module per cell",
    save="05_hotspot_module_assignment_umap.pdf",
    report="05_hotspot_module_assignment_umap.png",
)

### 10.5.3 - Comparison with an existing grouping

**What do I see?** For each group of `compare_column` (columns), the share of
its cells assigned to each module (rows). Every column sums up to 1.

**How can I interpret it?** A module that takes up almost all cells of one
group mirrors that group and adds little new information. A module that is
spread over several groups describes a program shared between them, which is
often the more interesting result. A group split between two modules contains
two states that the clustering did not separate.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Existing grouping in adata.obs to compare the module assignment with,
# e.g. a Leiden clustering.
compare_column = "leiden"

In [ ]:
with pd.option_context("display.float_format", "{:.2f}".format):
    display(pd.crosstab(
        adata_filtered.obs["hotspot_module_assignment"],
        adata_filtered.obs[compare_column],
        normalize="columns",
    ))

In [ ]:
# Module assignment next to the existing grouping
_ = pl.embedding.plot_embedding(
    adata_filtered,
    method="umap",
    color=["hotspot_module_assignment", compare_column],
)

## 11 - Module enrichment
<hr style="border:2px solid black"> </hr>

The modules from section 10 are sets of co-varying genes without a label. This
section tests each module against gene set libraries (Enrichr) to suggest what
the program might be biologically.

Note that this is the only step that depends on external annotation: results
are only as good as the coverage of the chosen library for your organism, and
the gene symbols in `adata.var_names` have to match those of the library. A
module without significant terms is not necessarily meaningless.

| Parameter | Description | Options | Default |
|:---|:---|:---|:---|
| `organism` | Organism of the dataset; selects the Enrichr database. Must match the gene symbols in `adata.var_names`. | `"human"` (*Homo sapiens*, [Enrichr](https://maayanlab.cloud/Enrichr/)), `"mouse"` (*Mus musculus*, [Enrichr](https://maayanlab.cloud/Enrichr/)), `"fish"` (zebrafish, *Danio rerio*, [FishEnrichr](https://maayanlab.cloud/FishEnrichr/)), `"fly"` (*Drosophila melanogaster*, [FlyEnrichr](https://maayanlab.cloud/FlyEnrichr/)), `"worm"` (*Caenorhabditis elegans*, [WormEnrichr](https://maayanlab.cloud/WormEnrichr/)), `"yeast"` (*Saccharomyces cerevisiae*, [YeastEnrichr](https://maayanlab.cloud/YeastEnrichr/)) | `"human"` |
| `gene_sets` | Enrichr libraries to test against. Availability differs per organism, see [the library list](https://maayanlab.cloud/Enrichr/#libraries). | List of library names | `["GO_Biological_Process_2018"]` |
| `min_module_size` | Minimum number of genes before a module is tested. | Integer | `5` |
| `overwrite` | Whether to replace enrichment results already stored in the object. | `True`, `False` | `True` |
| `enrichment_top_n` | Number of top terms shown per module in 11.2. Higher values make the plot taller and harder to read. | Integer | `5` |

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Organism used for the enrichment database, see the table above.
# "human" (Homo sapiens), "mouse" (Mus musculus), "fish" (zebrafish, Danio rerio),
# "fly" (Drosophila melanogaster), "worm" (Caenorhabditis elegans),
# "yeast" (Saccharomyces cerevisiae)
# Make sure this matches the gene symbols in adata.var_names.
organism = "human"

In [ ]:
# Libraries available for the selected organism.
# Copy the names you want into gene_sets.
gp.get_library_name(organism=organism)

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Gene set enrichment settings

# Gene set libraries used for enrichment analysis.
# Each Hotspot module will be tested against these gene set collections.
gene_sets = [
    "HDSigDB_Human_2021"
]

# Minimum number of genes required in a Hotspot module
# before enrichment analysis is performed for that module.
min_module_size = 5

# If True, overwrite existing enrichment results with the same key/output name.
overwrite = True

# Number of top enriched terms shown per module in the enrichment plot.
enrichment_top_n = 5

### 11.1 - Run module enrichment

Runs one Enrichr query per module. Modules smaller than `min_module_size` are
skipped, since a handful of genes cannot produce a reliable enrichment.

In [ ]:
hotspot.run_hotspot_module_enrichment(
    adata=adata_filtered,
    gene_sets=gene_sets,
    organism=organism,
    min_module_size=min_module_size,
    overwrite=overwrite,
    save_table="08_hotspot_module_enrichment.tsv",
)

### 11.2 - Plot module enrichment

The dotplot shows the top enriched terms per module, i.e. the biological
processes the genes of a module are associated with more often than expected
by chance.

**How to read it.** A module whose top terms all point in one direction — cell
cycle, extracellular matrix, immune response — has a clear interpretation, and
that is the result you are usually after. Terms that appear in several modules
at once are typically broad parent categories and carry little discriminative
information.

Absent terms are not evidence of absence. A module without enriched terms may
simply contain genes that the chosen library does not cover well, which is
common for non-model organisms and for anything outside the classic annotated
pathways. The same holds for the p-values: with small modules only a handful
of genes overlap a term, so significance is reached rarely even when the
association is real.

In [ ]:
top_terms = hotspot.plot_hotspot_module_enrichment(
    adata_filtered,
    top_n=enrichment_top_n,
    save="06_hotspot_module_enrichment.pdf",
    report="06_hotspot_module_enrichment.png",
)

### 11.3 - Top terms per module

The same terms as in the dotplot, with the numbers behind them. `Overlap` is
the number of module genes in the term relative to the term size, `Odds Ratio`
and `Combined Score` describe how strong the enrichment is, and `Genes` lists
the overlapping genes (in uppercase, since Enrichr matches gene symbols in uppercase).

The full table for all modules, not just the top terms, is written to
`08_hotspot_module_enrichment.tsv` in the table directory.

In [ ]:
with pd.option_context("display.max.rows", 100, "display.max.columns", None):
    display(top_terms)

## 12 - Save filtered adata
<hr style="border:2px solid black"> </hr>
Store the final results

In [ ]:
# Saving the data
utils.adata.save_h5ad(
    adata_filtered,
    "anndata_hotspot.h5ad",
    report=["hotspot_overview.md", "hotspot_obs.png", "hotspot_var.png"],
)

In [ ]:
sctoolbox.settings.close_logfile()